In [25]:
import pandas as pd

In [ ]:
def preprocess_peptide_dataset(csv_path, min_samples_per_cell_line=10, exclude_csv_path=None):
    """
    Предобработка датасета пептидов для soft-prompt tuning
    """
    # 1. Загрузка датасета
    print(f"Загрузка датасета из {csv_path}...")
    df = pd.read_csv(csv_path)
    print(f"Исходный размер датасета: {len(df)} строк")
    
    # 2. Оставляем только нужные колонки
    df = df[['sequence', 'is_cpp', 'cell_line']].copy()
    
    # 3. Приведение последовательностей к единому виду
    print("\nОбработка последовательностей...")
    df['sequence'] = df['sequence'].astype(str).str.upper()
    
    # Определяем разрешенные символы
    allowed_chars = set("ACDEFGHIKLMNPQRSTVWYXZBOU")
    
    def is_valid_sequence(seq, allowed_set):
        """Проверяет, состоит ли последовательность только из разрешенных символов."""
        if not isinstance(seq, str) or not seq:
            return False
        return all(char in allowed_set for char in seq)
    
    # Фильтрация последовательностей
    df['is_valid'] = df['sequence'].apply(lambda x: is_valid_sequence(x, allowed_chars))
    df_filtered = df[df['is_valid']].copy()
    df_filtered = df_filtered.drop(columns=['is_valid'])
    
    print(f"После фильтрации по допустимым символам: {len(df_filtered)} строк")
    print(f"Отфильтровано: {len(df) - len(df_filtered)} строк")
    
    # удаление последовательностей, совпадающих с df_2
    if exclude_csv_path:
        print(f"\nУдаление последовательностей, совпадающих с {exclude_csv_path}...")
        df_exclude = pd.read_csv(exclude_csv_path)

        if 'sequence' in df_exclude.columns:
            exclude_sequences = set(df_exclude['sequence'].astype(str))
            
            count_before = len(df_filtered)
            df_filtered = df_filtered[~df_filtered['sequence'].isin(exclude_sequences)].copy()
            excluded_count = count_before - len(df_filtered)

            print(f"Найдено и удалено совпадений: {excluded_count}")
            print(f"Осталось после удаления: {len(df_filtered)} строк")

        else:
            print(f"ВНИМАНИЕ: В файле {exclude_csv_path} нет колонки 'sequence'. Пропуск этапа.")

    # 4.  Заполнение NaN в cell_line
    # nan_count = df_filtered['cell_line'].isna().sum()
    # if nan_count > 0:
    #    print(f"\nОбнаружено {nan_count} строк без значения cell_line — заполняем значением 'unknown'.")
    #    df_filtered['cell_line'] = df_filtered['cell_line'].fillna('unknown')
    
    # 5. Статистика по CPP
    print("\nСтатистика по CPP:")
    cpp_counts = df_filtered['is_cpp'].value_counts()
    print(cpp_counts)

    # Удаление строк с NaN в cell_line
    nan_count = df_filtered['cell_line'].isna().sum()
    if nan_count > 0:
        print(f"\nОбнаружено {nan_count} строк без значения cell_line — удаляем.")
        df_filtered = df_filtered.dropna(subset=['cell_line']).copy()
        print(f"Осталось после удаления: {len(df_filtered)} строк")

    # 5. Статистика по CPP
    print("\nСтатистика по CPP:")
    cpp_counts = df_filtered['is_cpp'].value_counts()
    print(cpp_counts)

    # 6. Отсев по is_cpp — оставляем только CPP
    print("\nОтсев не-CPP пептидов...")
    cpp_df = df_filtered[df_filtered['is_cpp'] == 1].copy()
    print(f"CPP пептидов: {len(cpp_df)}")
    
    # 7. Распределение CPP по клеточным линиям ДО объединения редких

    # Подсчет образцов для каждой клеточной линии
    cell_line_counts = cpp_df['cell_line'].value_counts()
    print("\nРаспределение по клеточным линиям до объединения:")
    for cell_line, count in cell_line_counts.items():
        print(f"  {cell_line}: {count} образцов")
    
    # Определяем редкие клеточные линии
    print(f"\nОбработка клеточных линий (минимум образцов: {min_samples_per_cell_line})...")
    rare_cell_lines = cell_line_counts[cell_line_counts < min_samples_per_cell_line].index.tolist()
    
    if 'unknown' in rare_cell_lines:
        rare_cell_lines.remove('unknown')

    if rare_cell_lines:
        print(f"\nКлеточные линии с < {min_samples_per_cell_line} образцов (будут объединены в 'other'):")
        for cl in rare_cell_lines:
            print(f"  {cl}: {cell_line_counts[cl]} образцов")
        
        # Заменяем редкие клеточные линии на "other"
        cpp_df.loc[df_filtered['cell_line'].isin(rare_cell_lines), 'cell_line'] = 'other'
    
    # Финальное распределение
    final_cell_line_counts = cpp_df['cell_line'].value_counts()
    print("\nРаспределение по клеточным линиям:")
    for cell_line, count in final_cell_line_counts.items():
        print(f"  {cell_line}: {count} образцов")
    
    return cpp_df

def perform_oversampling(df, target_samples):
    """Выполняет оверсэмплинг для каждого класса до target_samples."""
    
    # Группируем по клеточным линиям
    grouped = df.groupby('cell_line')
    
    resampled_dfs = []
    for name, group in grouped:
        if name == 'unknown':
            print(f"Класс '{name}' не оверсэмплится (размер: {len(group)})")
            resampled_dfs.append(group)
            continue
    
        if len(group) < target_samples:
            # Дублируем случайные строки, чтобы достичь нужного количества
            resampled_group = group.sample(n=target_samples, replace=True, random_state=42)
            resampled_dfs.append(resampled_group)
        else:
            # Если класс уже достаточно большой, оставляем его как есть
            resampled_dfs.append(group)
            
    # Собираем все обратно в один датафрейм и перемешиваем
    df_resampled = pd.concat(resampled_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    return df_resampled

def format_sequences_for_protgpt2(df, seq_column='sequence'):
    """
    Преобразует последовательности в формат ProtGPT2:
    - Добавляет <|endoftext|> в начало
    - Добавляет переводы строк каждые 60 аминокислот
    
    Args:
        df: DataFrame с последовательностями
        seq_column: название колонки с последовательностями
    
    Returns:
        DataFrame с преобразованными последовательностями
    """
    df = df.copy()
    
    def format_sequence(seq):
        # Добавляем токен в начало
        formatted = "<|endoftext|>"
        
        # Разбиваем на строки по 60 символов
        for i in range(0, len(seq), 60):
            formatted += "\n" + seq[i:i+60]
        
        formatted += "\n<|endoftext|>"
        
        return formatted
    
    # Применяем форматирование
    df[seq_column] = df[seq_column].apply(format_sequence)
    
    return df

In [3]:
path = 'C:/Users/ALI/itmo-cpp/input_data/peptides_without_train_rows.csv'
exclude_path = 'C:/Users/ALI/itmo-cpp/input_data/train_rows_for_active_sampling.csv'

In [35]:
processed_df = preprocess_peptide_dataset(path, min_samples_per_cell_line=10, exclude_csv_path=exclude_path)

Загрузка датасета из C:/Users/ALI/itmo-cpp/input_data/peptides_without_train_rows.csv...
Исходный размер датасета: 2422 строк

Обработка последовательностей...
После фильтрации по допустимым символам: 2182 строк
Отфильтровано: 240 строк

Удаление последовательностей, совпадающих с C:/Users/ALI/itmo-cpp/input_data/train_rows_for_active_sampling.csv...
Найдено и удалено совпадений: 11
Осталось после удаления: 2171 строк

Обнаружено 1522 строк без значения cell_line — удаляем.
Осталось после удаления: 649 строк

Статистика по CPP:
is_cpp
True     559
False     90
Name: count, dtype: int64

Отсев не-CPP пептидов...
CPP пептидов: 559

Распределение по клеточным линиям до объединения:
  HeLa cells: 142 образцов
  NIH-3T3 cells: 48 образцов
  A549 cells: 34 образцов
  CHO-K1 cells: 28 образцов
  MDA-MB-231 cells: 27 образцов
  CHO cells: 26 образцов
  MCF7 cells: 23 образцов
  HaCaT cells: 22 образцов
  HEK293 cells: 17 образцов
  U87 cells: 15 образцов
  Human bowes melanoma cells: 12 образц

In [36]:
cpp_df = processed_df[processed_df['is_cpp'] == 1].copy()

# Определяем целевое количество сэмплов
target_count = cpp_df[cpp_df['cell_line'] != 'unknown']['cell_line'].value_counts().max()
print(f"\nВыполняется оверсэмплинг минорных классов до {target_count} образцов...")

# Применяем оверсэмплинг
df_train_balanced = perform_oversampling(cpp_df, target_count)

print("\nРазмер датасета после оверсэмплинга:")
print(df_train_balanced['cell_line'].value_counts())
print(f"Итоговый размер обучающего датасета: {len(df_train_balanced)} строк")


Выполняется оверсэмплинг минорных классов до 165 образцов...

Размер датасета после оверсэмплинга:
cell_line
MDA-MB-231 cells              165
A549 cells                    165
CHO-K1 cells                  165
U87 cells                     165
Human bowes melanoma cells    165
CHO cells                     165
MCF7 cells                    165
HeLa cells                    165
HEK293 cells                  165
NIH-3T3 cells                 165
HaCaT cells                   165
other                         165
Name: count, dtype: int64
Итоговый размер обучающего датасета: 1980 строк


In [37]:
cpp_df.to_csv('cpp_without_train_rows_165_no_unknown.csv', index=False)

In [38]:
# Форматируем для ProtGPT2
df_train_balanced = format_sequences_for_protgpt2(df_train_balanced)

In [39]:
df_train_balanced.to_csv('cpp_only_165_no_unknown.csv', index=False)